## Google Colab Setup

Runtime: A100 GPU (recommended)
  Go to: Runtime > Change runtime type > Hardware accelerator > A100

If A100 is unavailable, use L4. Avoid T4 for this pipeline — SAM is slow on T4.

Mount Google Drive before running (prevents data loss on session disconnect)

In [ ]:
  from google.colab import drive
  drive.mount('/content/drive')
  SAVE_DIR = '/content/drive/MyDrive/dog_breed_pipeline/'

## Installations

## Cell 1 — PyTorch (usually pre-installed on Colab, verify version):

In [ ]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

## CELL 2 — Install SAM

In [ ]:
# Run this cell once per session
!pip install segment-anything
!wget https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth



## CELL 3 — Imports

In [ ]:

import torch
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader, random_split

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    silhouette_score,
    adjusted_rand_score,
    normalized_mutual_info_score,
    confusion_matrix
)
from sklearn.manifold import TSNE
from tqdm import tqdm
from PIL import Image
from collections import defaultdict
import pandas as pd
import time
import warnings
warnings.filterwarnings('ignore')

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch version : {torch.__version__}")
print(f"Device          : {device}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")


## CELL 4 — Download and Load Imagewoof

In [ ]:
# --- Download ---
!wget https://s3.amazonaws.com/fast-ai-imageclas/imagewoof2-160.tgz
!tar -xzf imagewoof2-160.tgz



In [ ]:
# Breed name mapping (folder name -> readable name)
BREED_NAMES = {
    'n02086240': 'Shih-Tzu',
    'n02087394': 'Rhodesian Ridgeback',
    'n02088364': 'Beagle',
    'n02089973': 'English Foxhound',
    'n02093754': 'Australian Terrier',
    'n02096294': 'Border Terrier',
    'n02099601': 'Golden Retriever',
    'n02105641': 'Old English Sheepdog',
    'n02111889': 'Samoyed',
    'n02115641': 'Shiba Inu'
}

# Transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Load dataset
full_dataset = torchvision.datasets.ImageFolder(
    root='./imagewoof2-160/train',
    transform=transform
)

readable_names = [
    BREED_NAMES.get(c, c) for c in full_dataset.classes
]

print(f"Total images  : {len(full_dataset)}")
print(f"Breeds        : {len(full_dataset.classes)}")
print()
for i, name in enumerate(readable_names):
    count = sum(1 for _, label in full_dataset if label == i)
    print(f"  {i:2d}. {name:<25} {count} images")



## CELL 5 — Split Dataset and Create DataLoaders

In [ ]:

train_size = int(0.8 * len(full_dataset))
val_size   = int(0.1 * len(full_dataset))
test_size  = len(full_dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    full_dataset,
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, num_workers=2)

print(f"Training   : {len(train_dataset)} images ({len(train_loader)} batches)")
print(f"Validation : {len(val_dataset)} images ({len(val_loader)} batches)")
print(f"Test       : {len(test_dataset)} images ({len(test_loader)} batches)")


## CELL 6 — Load ResNet-18 Feature Extractor

In [ ]:
resnet18 = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
resnet18 = torch.nn.Sequential(*list(resnet18.children())[:-1])
resnet18.eval()
resnet18 = resnet18.to(device)

# Verify output shape
dummy = torch.randn(1, 3, 224, 224).to(device)
with torch.no_grad():
    out = resnet18(dummy)
print(f"ResNet-18 loaded")
print(f"Output shape : {out.shape}  (expected [1, 512, 1, 1])")
print(f"Parameters   : {sum(p.numel() for p in resnet18.parameters()):,}")


## CELL 7 — Load SAM

In [ ]:

from segment_anything import sam_model_registry, SamPredictor

sam = sam_model_registry["vit_b"](checkpoint="sam_vit_b_01ec64.pth")
sam = sam.to(device)
predictor = SamPredictor(sam)

print(f"SAM ViT-B loaded on {device}")


## CELL 8 — Define Segmentation + Feature Extraction Function

In [ ]:

def segment_and_extract(model, dataset, predictor, device, description="Extracting"):
    """
    Single-pass extraction for both black and white backgrounds.
    SAM runs once per image. Features are extracted for both fill values
    in a single ResNet forward pass (batch of 2).

    Returns:
        features_baseline : (N, 512) — no segmentation
        features_black    : (N, 512) — black background
        features_white    : (N, 512) — white background
        labels            : (N,)     — true breed indices
    """
    model.eval()

    feat_baseline = []
    feat_black    = []
    feat_white    = []
    labels_list   = []
    coverage_warnings = []

    for idx in tqdm(range(len(dataset)), desc=description):
        try:
            image, label = dataset[idx]

            # ---- Baseline: extract from raw image ----
            with torch.no_grad():
                raw_input = image.unsqueeze(0).to(device)
                raw_feat = model(raw_input).view(1, -1).cpu().numpy()
            feat_baseline.append(raw_feat)

            # ---- SAM segmentation ----
            # Denormalize to uint8 for SAM
            image_np = image.permute(1, 2, 0).numpy()
            image_np = (image_np * 0.5) + 0.5
            image_np = np.clip(image_np, 0, 1)
            image_uint8 = (image_np * 255).astype(np.uint8)

            predictor.set_image(image_uint8)

            # Robust multi-point prompt
            h, w = image_uint8.shape[:2]
            prompt_points = np.array([
                [w // 2,       h // 2      ],
                [w // 2,       h // 3      ],
                [w // 2,       2 * h // 3  ],
                [w // 3,       h // 2      ],
                [2 * w // 3,   h // 2      ],
            ])
            prompt_labels = np.ones(len(prompt_points), dtype=int)

            masks, scores, _ = predictor.predict(
                point_coords=prompt_points,
                point_labels=prompt_labels,
                multimask_output=True
            )

            # Pick mask with coverage between 10% and 80%
            mask = None
            best_score = -1
            for m, s in zip(masks, scores):
                coverage = m.mean()
                if 0.10 <= coverage <= 0.80 and s > best_score:
                    mask = m
                    best_score = s

            # Fall back to highest confidence mask if none passed the filter
            if mask is None:
                mask = masks[np.argmax(scores)]

            # Coverage warning
            coverage = mask.mean()
            if coverage < 0.05 or coverage > 0.95:
                coverage_warnings.append((idx, coverage))

            # ---- Apply mask: black and white ----
            mask_3d = mask[:, :, np.newaxis]  # (H, W, 1)

            img_black = np.where(mask_3d, image_np, 0.0)
            img_white = np.where(mask_3d, image_np, 1.0)

            # Re-normalize to match ResNet input format
            img_black = (img_black - 0.5) / 0.5
            img_white = (img_white - 0.5) / 0.5

            t_black = torch.tensor(img_black, dtype=torch.float32).permute(2, 0, 1)
            t_white = torch.tensor(img_white, dtype=torch.float32).permute(2, 0, 1)

            # ---- Extract features for both in one batch ----
            batch = torch.stack([t_black, t_white]).to(device)
            with torch.no_grad():
                feats = model(batch).view(2, -1).cpu().numpy()

            feat_black.append(feats[0:1])
            feat_white.append(feats[1:2])
            labels_list.append(label)

        except Exception as e:
            print(f"  Failed on index {idx}: {e}")
            feat_baseline.append(np.zeros((1, 512)))
            feat_black.append(np.zeros((1, 512)))
            feat_white.append(np.zeros((1, 512)))
            labels_list.append(label)

    if coverage_warnings:
        print(f"\n  {len(coverage_warnings)} images with unusual mask coverage:")
        for idx, cov in coverage_warnings[:5]:
            print(f"    Image {idx}: {cov*100:.1f}%")

    return (
        np.concatenate(feat_baseline, axis=0),
        np.concatenate(feat_black,    axis=0),
        np.concatenate(feat_white,    axis=0),
        np.array(labels_list)
    )



## CELL 9 — Extract Features (All Three Pipelines, Single Pass)

In [ ]:
import os
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Save directory ready: {SAVE_DIR}")

print("="*60)
print("Feature Extraction — Training Set")
print("="*60)
t0 = time.time()

train_feat_base, train_feat_black, train_feat_white, train_labels = \
    segment_and_extract(resnet18, train_dataset, predictor, device, "Training set")

train_time = time.time() - t0
print(f"\nTraining extraction time: {train_time/60:.1f} minutes")

print("\nFeature Extraction — Validation Set")
val_feat_base, val_feat_black, val_feat_white, val_labels = \
    segment_and_extract(resnet18, val_dataset, predictor, device, "Validation set")

# Shape verification
print(f"\nBaseline  — Train: {train_feat_base.shape},  Val: {val_feat_base.shape}")
print(f"Black bg  — Train: {train_feat_black.shape}, Val: {val_feat_black.shape}")
print(f"White bg  — Train: {train_feat_white.shape}, Val: {val_feat_white.shape}")

# Save to Drive immediately
print("\nSaving features to Google Drive...")
np.save(SAVE_DIR + 'train_feat_baseline.npy', train_feat_base)
np.save(SAVE_DIR + 'train_feat_black.npy',    train_feat_black)
np.save(SAVE_DIR + 'train_feat_white.npy',    train_feat_white)
np.save(SAVE_DIR + 'val_feat_baseline.npy',   val_feat_base)
np.save(SAVE_DIR + 'val_feat_black.npy',      val_feat_black)
np.save(SAVE_DIR + 'val_feat_white.npy',      val_feat_white)
np.save(SAVE_DIR + 'train_labels.npy',        train_labels)
np.save(SAVE_DIR + 'val_labels.npy',          val_labels)
print("Saved.")



## CELL 10 — Normalize Features

In [ ]:
def normalize(train_feat, val_feat):
    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train_feat)
    val_scaled   = scaler.transform(val_feat)
    return train_scaled, val_scaled, scaler

train_base_s,  val_base_s,  _ = normalize(train_feat_base,  val_feat_base)
train_black_s, val_black_s, _ = normalize(train_feat_black, val_feat_black)
train_white_s, val_white_s, _ = normalize(train_feat_white, val_feat_white)

print("Normalization complete")
print(f"  Baseline  — mean: {train_base_s.mean():.4f},  std: {train_base_s.std():.4f}")
print(f"  Black bg  — mean: {train_black_s.mean():.4f}, std: {train_black_s.std():.4f}")
print(f"  White bg  — mean: {train_white_s.mean():.4f}, std: {train_white_s.std():.4f}")


## # CELL 11 — K-Means Clustering (All Three Pipelines)

In [ ]:

def run_kmeans(train_features, val_features, train_labels, val_labels,
               pipeline_name, n_breeds=10):
    """
    Run K-Means over a range of K values, select best K by silhouette score,
    evaluate against true labels, and return results dictionary.
    """
    print(f"\n{'='*55}")
    print(f"Pipeline: {pipeline_name}")
    print(f"{'='*55}")

    k_range = [5, 8, 10, 12, 15, 20]
    sil_scores = []

    print("Testing K values:")
    for k in k_range:
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        clusters = km.fit_predict(train_features)
        sil = silhouette_score(train_features, clusters)
        sil_scores.append(sil)
        print(f"  K={k:2d}  Silhouette={sil:.4f}")

    best_k   = k_range[np.argmax(sil_scores)]
    best_sil = max(sil_scores)
    print(f"\nBest K: {best_k}  (Silhouette: {best_sil:.4f})")

    # Final clustering with best K
    final_km = KMeans(n_clusters=best_k, random_state=42, n_init=10)
    train_clusters = final_km.fit_predict(train_features)
    val_clusters   = final_km.predict(val_features)

    # If best K differs from true breed count, also run with n_breeds
    if best_k != n_breeds:
        km_true = KMeans(n_clusters=n_breeds, random_state=42, n_init=10)
        clusters_true = km_true.fit_predict(train_features)
        sil_true = silhouette_score(train_features, clusters_true)
        print(f"\nK={n_breeds} (true breed count)  Silhouette={sil_true:.4f}")

    # Evaluate on validation set
    val_sil = silhouette_score(val_features, val_clusters)
    ari     = adjusted_rand_score(val_labels, val_clusters)
    nmi     = normalized_mutual_info_score(val_labels, val_clusters)

    print(f"\nValidation metrics:")
    print(f"  Silhouette : {val_sil:.4f}")
    print(f"  ARI        : {ari:.4f}")
    print(f"  NMI        : {nmi:.4f}")

    return {
        'name'          : pipeline_name,
        'best_k'        : best_k,
        'train_clusters': train_clusters,
        'val_clusters'  : val_clusters,
        'final_km'      : final_km,
        'silhouette'    : val_sil,
        'ari'           : ari,
        'nmi'           : nmi,
    }


results_base  = run_kmeans(train_base_s,  val_base_s,  train_labels, val_labels, "Baseline (no SAM)")
results_black = run_kmeans(train_black_s, val_black_s, train_labels, val_labels, "SAM + Black Background")
results_white = run_kmeans(train_white_s, val_white_s, train_labels, val_labels, "SAM + White Background")


## # CELL 12 — Comparison Table

In [ ]:

print("\n" + "="*65)
print("Pipeline Comparison")
print("="*65)
print(f"{'Pipeline':<28} {'Best K':>6} {'Silhouette':>12} {'ARI':>8} {'NMI':>8}")
print("-"*65)
for r in [results_base, results_black, results_white]:
    print(f"{r['name']:<28} {r['best_k']:>6} {r['silhouette']:>12.4f} {r['ari']:>8.4f} {r['nmi']:>8.4f}")
print("="*65)

# Save comparison to Drive
comparison_df = pd.DataFrame([
    {
        'Pipeline'  : r['name'],
        'Best K'    : r['best_k'],
        'Silhouette': round(r['silhouette'], 4),
        'ARI'       : round(r['ari'], 4),
        'NMI'       : round(r['nmi'], 4),
    }
    for r in [results_base, results_black, results_white]
])
comparison_df.to_csv(SAVE_DIR + 'pipeline_comparison.csv', index=False)
print("\nSaved: pipeline_comparison.csv")



## CELL 13 — Purity Analysis

In [ ]:
def purity_analysis(train_clusters, train_labels, readable_names, pipeline_name, n_clusters):
    print(f"\n{'='*65}")
    print(f"Purity Analysis — {pipeline_name}")
    print(f"{'='*65}")
    print(f"{'Breed':<25} {'Dom. Cluster':>12} {'Correct':>8} {'Total':>7} {'Purity':>8}  Status")
    print("-"*65)

    cm = confusion_matrix(train_labels, train_clusters)
    purity_scores = []
    dominant_clusters = []

    for breed_idx, name in enumerate(readable_names):
        row   = cm[breed_idx]
        total = row.sum()
        dom   = row.argmax()
        correct = row[dom]
        purity  = correct / total
        purity_scores.append(purity)
        dominant_clusters.append(dom)

        if purity >= 0.80:
            status = "Excellent"
        elif purity >= 0.65:
            status = "Good"
        elif purity >= 0.50:
            status = "Moderate"
        else:
            status = "Poor"

        print(f"{name:<25} {dom:>12} {correct:>8} {total:>7} {purity:>7.1%}  {status}")

    print("-"*65)
    print(f"{'Average':<25} {'':>12} {'':>8} {'':>7} {np.mean(purity_scores):>7.1%}")

    # Shared cluster check
    cluster_to_breeds = defaultdict(list)
    for breed_idx, cluster in enumerate(dominant_clusters):
        cluster_to_breeds[cluster].append(readable_names[breed_idx])

    print(f"\nCluster Sharing Check:")
    found = False
    for cluster, breeds in cluster_to_breeds.items():
        if len(breeds) > 1:
            found = True
            print(f"  Cluster {cluster} shared by: {' & '.join(breeds)}")
    if not found:
        print("  Every breed has its own dominant cluster.")

    return cm


cm_base  = purity_analysis(results_base['train_clusters'],  train_labels, readable_names, "Baseline",        results_base['best_k'])
cm_black = purity_analysis(results_black['train_clusters'], train_labels, readable_names, "Black Background", results_black['best_k'])
cm_white = purity_analysis(results_white['train_clusters'], train_labels, readable_names, "White Background", results_white['best_k'])


## CELL 14 — Confusion Matrix Heatmap (for each pipeline)

In [ ]:
def plot_confusion_heatmap(cm, readable_names, n_clusters, pipeline_name, save_path):
    fig, ax = plt.subplots(figsize=(13, 8))

    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    im = ax.imshow(cm_norm, cmap='Blues', aspect='auto', vmin=0, vmax=1)
    plt.colorbar(im, ax=ax, label='Proportion of breed in cluster')

    ax.set_xticks(range(n_clusters))
    ax.set_xticklabels([f'C{i}' for i in range(n_clusters)], rotation=45, ha='right')
    ax.set_yticks(range(len(readable_names)))
    ax.set_yticklabels(readable_names)

    for i in range(len(readable_names)):
        for j in range(n_clusters):
            count = cm[i, j]
            if count > 0:
                color = 'white' if cm_norm[i, j] > 0.5 else 'black'
                ax.text(j, i, str(count), ha='center', va='center', fontsize=7, color=color)

    ax.set_xlabel('K-Means Cluster')
    ax.set_ylabel('True Breed')
    ax.set_title(f'Confusion Matrix — {pipeline_name}\n(color = proportion of breed in cluster)')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {save_path}")


plot_confusion_heatmap(cm_base,  readable_names, results_base['best_k'],  "Baseline",        SAVE_DIR + 'cm_baseline.png')
plot_confusion_heatmap(cm_black, readable_names, results_black['best_k'], "Black Background", SAVE_DIR + 'cm_black.png')
plot_confusion_heatmap(cm_white, readable_names, results_white['best_k'], "White Background", SAVE_DIR + 'cm_white.png')


## CELL 15 — t-SNE Visualization (for each pipeline)

In [ ]:

def plot_tsne(train_features_scaled, train_labels, train_clusters,
              readable_names, best_k, pipeline_name, save_path):

    print(f"Running t-SNE for {pipeline_name}...")
    tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
    features_2d = tsne.fit_transform(train_features_scaled)
    print("t-SNE complete.")

    colors = plt.cm.tab10(np.linspace(0, 1, 10))
    fig, axes = plt.subplots(1, 2, figsize=(20, 8))

    # Left: true breed labels
    for i, name in enumerate(readable_names):
        mask = train_labels == i
        axes[0].scatter(features_2d[mask, 0], features_2d[mask, 1],
                        c=[colors[i % 10]], label=name, alpha=0.6, s=12)
    axes[0].set_title('True Breed Labels', fontsize=13, fontweight='bold')
    axes[0].legend(fontsize=7, markerscale=2)
    axes[0].set_xlabel('t-SNE Dimension 1')
    axes[0].set_ylabel('t-SNE Dimension 2')

    # Right: K-Means clusters
    for i in range(best_k):
        mask = train_clusters == i
        axes[1].scatter(features_2d[mask, 0], features_2d[mask, 1],
                        c=[colors[i % 10]], label=f'Cluster {i}', alpha=0.6, s=12)
    axes[1].set_title(f'K-Means Clusters (K={best_k})', fontsize=13, fontweight='bold')
    axes[1].legend(fontsize=7, markerscale=2)
    axes[1].set_xlabel('t-SNE Dimension 1')
    axes[1].set_ylabel('t-SNE Dimension 2')

    plt.suptitle(f't-SNE Visualization — {pipeline_name}', fontsize=14)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {save_path}")


plot_tsne(train_base_s,  train_labels, results_base['train_clusters'],
          readable_names, results_base['best_k'],
          "Baseline (No Segmentation)", SAVE_DIR + 'tsne_baseline.png')

plot_tsne(train_black_s, train_labels, results_black['train_clusters'],
          readable_names, results_black['best_k'],
          "SAM + Black Background", SAVE_DIR + 'tsne_black.png')

plot_tsne(train_white_s, train_labels, results_white['train_clusters'],
          readable_names, results_white['best_k'],
          "SAM + White Background", SAVE_DIR + 'tsne_white.png')



## CELL 16 — Cluster Sample Images

In [ ]:

def plot_cluster_samples(dataset, train_clusters, train_labels,
                         readable_names, best_k, pipeline_name,
                         save_path, samples_per_cluster=5):

    fig, axes = plt.subplots(best_k, samples_per_cluster,
                             figsize=(15, 3 * best_k))

    for cluster_id in range(best_k):
        cluster_indices = np.where(train_clusters == cluster_id)[0]
        if len(cluster_indices) == 0:
            continue

        cluster_true_labels = train_labels[cluster_indices]
        dominant_breed_idx  = np.bincount(cluster_true_labels).argmax()
        dominant_breed      = readable_names[dominant_breed_idx]
        purity = (cluster_true_labels == dominant_breed_idx).sum() / len(cluster_true_labels)

        sample_indices = np.random.choice(
            cluster_indices,
            min(samples_per_cluster, len(cluster_indices)),
            replace=False
        )

        for j, idx in enumerate(sample_indices):
            image, label = dataset[idx]
            image = image.permute(1, 2, 0).numpy()
            image = (image * 0.5) + 0.5
            image = np.clip(image, 0, 1)

            axes[cluster_id, j].imshow(image)
            axes[cluster_id, j].axis('off')
            axes[cluster_id, j].set_title(readable_names[train_labels[idx]], fontsize=7)

        axes[cluster_id, 0].set_ylabel(
            f'Cluster {cluster_id}\n{dominant_breed}\n({purity:.0%})',
            fontsize=8, rotation=0, labelpad=85, va='center'
        )

    plt.suptitle(f'Cluster Samples — {pipeline_name}\n(title = true breed)', fontsize=13)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {save_path}")


plot_cluster_samples(train_dataset, results_base['train_clusters'],  train_labels,
                     readable_names, results_base['best_k'],
                     "Baseline", SAVE_DIR + 'samples_baseline.png')

plot_cluster_samples(train_dataset, results_black['train_clusters'], train_labels,
                     readable_names, results_black['best_k'],
                     "SAM + Black Background", SAVE_DIR + 'samples_black.png')

plot_cluster_samples(train_dataset, results_white['train_clusters'], train_labels,
                     readable_names, results_white['best_k'],
                     "SAM + White Background", SAVE_DIR + 'samples_white.png')



## CELL 17 — Stanford Dogs (Scalability Test)

In [ ]:
!wget http://vision.stanford.edu/aditya86/ImageNetDogs/images.tar
!tar -xf images.tar

In [ ]:

stanford_dataset = torchvision.datasets.ImageFolder(
    root='./Images',
    transform=transform
)

print(f"Stanford Dogs loaded")
print(f"Total images : {len(stanford_dataset)}")
print(f"Breeds       : {len(stanford_dataset.classes)}")

# Split
s_train_size = int(0.8 * len(stanford_dataset))
s_val_size   = int(0.1 * len(stanford_dataset))
s_test_size  = len(stanford_dataset) - s_train_size - s_val_size

s_train, s_val, s_test = random_split(
    stanford_dataset,
    [s_train_size, s_val_size, s_test_size],
    generator=torch.Generator().manual_seed(42)
)

print(f"\nTrain : {len(s_train)}")
print(f"Val   : {len(s_val)}")
print(f"Test  : {len(s_test)}")

# Extract features — Stanford Dogs (all three pipelines, single pass)
print("\nExtracting Stanford Dogs features...")

s_train_feat_base, s_train_feat_black, s_train_feat_white, s_train_labels = \
    segment_and_extract(resnet18, s_train, predictor, device, "Stanford Train")

s_val_feat_base, s_val_feat_black, s_val_feat_white, s_val_labels = \
    segment_and_extract(resnet18, s_val, predictor, device, "Stanford Val")

# Save Stanford features
np.save(SAVE_DIR + 'stanford_train_feat_baseline.npy', s_train_feat_base)
np.save(SAVE_DIR + 'stanford_train_feat_black.npy',    s_train_feat_black)
np.save(SAVE_DIR + 'stanford_train_feat_white.npy',    s_train_feat_white)
np.save(SAVE_DIR + 'stanford_val_feat_baseline.npy',   s_val_feat_base)
np.save(SAVE_DIR + 'stanford_val_feat_black.npy',      s_val_feat_black)
np.save(SAVE_DIR + 'stanford_val_feat_white.npy',      s_val_feat_white)
np.save(SAVE_DIR + 'stanford_train_labels.npy',        s_train_labels)
np.save(SAVE_DIR + 'stanford_val_labels.npy',          s_val_labels)
print("Stanford features saved.")

# Normalize
s_train_base_s,  s_val_base_s,  _ = normalize(s_train_feat_base,  s_val_feat_base)
s_train_black_s, s_val_black_s, _ = normalize(s_train_feat_black, s_val_feat_black)
s_train_white_s, s_val_white_s, _ = normalize(s_train_feat_white, s_val_feat_white)

# Cluster — Stanford Dogs (n_breeds=120)
s_results_base  = run_kmeans(s_train_base_s,  s_val_base_s,  s_train_labels, s_val_labels, "Stanford Baseline",   n_breeds=120)
s_results_black = run_kmeans(s_train_black_s, s_val_black_s, s_train_labels, s_val_labels, "Stanford Black bg",   n_breeds=120)
s_results_white = run_kmeans(s_train_white_s, s_val_white_s, s_train_labels, s_val_labels, "Stanford White bg",   n_breeds=120)

# Final comparison table
print("\n" + "="*75)
print("Final Comparison — Imagewoof (10 breeds) vs Stanford Dogs (120 breeds)")
print("="*75)
print(f"{'Pipeline':<35} {'Dataset':<15} {'Best K':>6} {'ARI':>8} {'NMI':>8}")
print("-"*75)
for r, ds in [
    (results_base,    'Imagewoof'),
    (results_black,   'Imagewoof'),
    (results_white,   'Imagewoof'),
    (s_results_base,  'Stanford'),
    (s_results_black, 'Stanford'),
    (s_results_white, 'Stanford'),
]:
    print(f"{r['name']:<35} {ds:<15} {r['best_k']:>6} {r['ari']:>8.4f} {r['nmi']:>8.4f}")
print("="*75)

## Cell A — Load Oxford Pets and filter to dogs and cats separately:

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
import torchvision.datasets as datasets
from torch.utils.data import DataLoader, random_split, ConcatDataset, Subset
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score
from sklearn.manifold import TSNE
from tqdm import tqdm
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/dog_breed_pipeline/'
print(f"Save dir: {SAVE_DIR}")
print(f"Files available: {os.listdir(SAVE_DIR)}")

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

full_pets = datasets.OxfordIIITPet(
    root='./data', split='trainval',
    target_types='category',
    transform=transform, download=True
)
test_pets = datasets.OxfordIIITPet(
    root='./data', split='test',
    target_types='category',
    transform=transform, download=True
)

print(f"Full pets: {len(full_pets)}")
print(f"Test pets: {len(test_pets)}")

In [ ]:
# ============================================
# OXFORD PETS — LOAD AND FILTER (FIXED)
# ============================================

import numpy as np
from torch.utils.data import ConcatDataset, Subset

# These match exactly what torchvision reports
CAT_CLASSES = [
    'Abyssinian', 'Bengal', 'Birman', 'Bombay', 'British Shorthair',
    'Egyptian Mau', 'Maine Coon', 'Persian', 'Ragdoll',
    'Russian Blue', 'Siamese', 'Sphynx'
]

DOG_CLASSES = [
    'American Bulldog', 'American Pit Bull Terrier', 'Basset Hound',
    'Beagle', 'Boxer', 'Chihuahua', 'English Cocker Spaniel',
    'English Setter', 'German Shorthaired', 'Great Pyrenees', 'Havanese',
    'Japanese Chin', 'Keeshond', 'Leonberger', 'Miniature Pinscher',
    'Newfoundland', 'Pomeranian', 'Pug', 'Saint Bernard', 'Samoyed',
    'Scottish Terrier', 'Shiba Inu', 'Staffordshire Bull Terrier',
    'Wheaten Terrier', 'Yorkshire Terrier'
]

class_names = full_pets.classes

# Find label indices for each species
cat_label_indices = [i for i, c in enumerate(class_names) if c in CAT_CLASSES]
dog_label_indices = [i for i, c in enumerate(class_names) if c in DOG_CLASSES]

print(f"Cat classes found : {len(cat_label_indices)}  {[class_names[i] for i in cat_label_indices]}")
print(f"Dog classes found : {len(dog_label_indices)}  {[class_names[i] for i in dog_label_indices]}")

# Combine trainval + test into one pool
all_pets = ConcatDataset([full_pets, test_pets])

# Get all labels
all_labels = np.array(
    [full_pets[i][1] for i in range(len(full_pets))] +
    [test_pets[i][1]  for i in range(len(test_pets))]
)

# Separate indices by species
cat_indices = np.where(np.isin(all_labels, cat_label_indices))[0]
dog_indices = np.where(np.isin(all_labels, dog_label_indices))[0]

print(f"\nTotal cat images : {len(cat_indices)}")
print(f"Total dog images : {len(dog_indices)}")

# Remap labels to 0-based within each species
cat_label_map = {orig: new for new, orig in enumerate(sorted(cat_label_indices))}
dog_label_map = {orig: new for new, orig in enumerate(sorted(dog_label_indices))}

cat_readable = [class_names[i] for i in sorted(cat_label_indices)]
dog_readable = [class_names[i] for i in sorted(dog_label_indices)]

print(f"\nCat breeds ({len(cat_readable)}): {cat_readable}")
print(f"\nDog breeds ({len(dog_readable)}): {dog_readable}")

## Cell B — Split Oxford Dogs and Cats into train/val/test:


In [ ]:
# ============================================
# OXFORD PETS — SPLIT INTO TRAIN/VAL/TEST
# ============================================

from torch.utils.data import DataLoader
import torch

def split_species_dataset(all_dataset, species_indices, label_map, seed=42):
    """
    Takes a subset of Oxford Pets by species indices,
    splits 80/10/10, returns datasets and remapped labels.
    """
    # Create subset
    subset = Subset(all_dataset, species_indices)

    # Split
    n = len(subset)
    train_size = int(0.8 * n)
    val_size   = int(0.1 * n)
    test_size  = n - train_size - val_size

    train_ds, val_ds, test_ds = torch.utils.data.random_split(
        subset,
        [train_size, val_size, test_size],
        generator=torch.Generator().manual_seed(seed)
    )
    return train_ds, val_ds, test_ds


# Split dogs
dog_train, dog_val, dog_test = split_species_dataset(
    all_pets, dog_indices, dog_label_map
)

# Split cats
cat_train, cat_val, cat_test = split_species_dataset(
    all_pets, cat_indices, cat_label_map
)

print("Oxford Dogs split:")
print(f"  Train : {len(dog_train)}")
print(f"  Val   : {len(dog_val)}")
print(f"  Test  : {len(dog_test)}")

print("\nOxford Cats split:")
print(f"  Train : {len(cat_train)}")
print(f"  Val   : {len(cat_val)}")
print(f"  Test  : {len(cat_test)}")

# DataLoaders
batch_size = 32

dog_train_loader = DataLoader(dog_train, batch_size=batch_size, shuffle=False, num_workers=2)
dog_val_loader   = DataLoader(dog_val,   batch_size=batch_size, shuffle=False, num_workers=2)
cat_train_loader = DataLoader(cat_train, batch_size=batch_size, shuffle=False, num_workers=2)
cat_val_loader   = DataLoader(cat_val,   batch_size=batch_size, shuffle=False, num_workers=2)

print("\nDataLoaders ready.")

## Cell C — Extract labels correctly (remapped to 0-based):


In [ ]:
# ============================================
# HELPER — GET REMAPPED LABELS FOR A SPLIT
# ============================================

def get_remapped_labels(dataset, label_map):
    """
    Extract true labels from a dataset split and remap
    them to 0-based indices within that species.
    """
    labels = []
    for i in range(len(dataset)):
        _, orig_label = dataset[i]
        labels.append(label_map[orig_label])
    return np.array(labels)

print("Extracting remapped labels...")
dog_train_labels = get_remapped_labels(dog_train, dog_label_map)
dog_val_labels   = get_remapped_labels(dog_val,   dog_label_map)
cat_train_labels = get_remapped_labels(cat_train, cat_label_map)
cat_val_labels   = get_remapped_labels(cat_val,   cat_label_map)

print(f"Dog train labels shape : {dog_train_labels.shape}  unique: {np.unique(dog_train_labels)}")
print(f"Cat train labels shape : {cat_train_labels.shape}  unique: {np.unique(cat_train_labels)}")

## Cell D — Extract features for all three pipelines (dogs and cats):


In [ ]:
# ============================================
# FEATURE EXTRACTION — OXFORD DOGS AND CATS
# ============================================

print("="*55)
print("Oxford Dogs — Feature Extraction")
print("="*55)

dog_train_base, dog_train_black, dog_train_white, _ = \
    segment_and_extract(resnet18, dog_train, predictor, device, "Dog Train")

dog_val_base, dog_val_black, dog_val_white, _ = \
    segment_and_extract(resnet18, dog_val, predictor, device, "Dog Val")

# Save dogs immediately
np.save(SAVE_DIR + 'oxford_dog_train_base.npy',  dog_train_base)
np.save(SAVE_DIR + 'oxford_dog_train_black.npy', dog_train_black)
np.save(SAVE_DIR + 'oxford_dog_train_white.npy', dog_train_white)
np.save(SAVE_DIR + 'oxford_dog_val_base.npy',    dog_val_base)
np.save(SAVE_DIR + 'oxford_dog_val_black.npy',   dog_val_black)
np.save(SAVE_DIR + 'oxford_dog_val_white.npy',   dog_val_white)
np.save(SAVE_DIR + 'oxford_dog_train_labels.npy', dog_train_labels)
np.save(SAVE_DIR + 'oxford_dog_val_labels.npy',   dog_val_labels)
print("Oxford Dogs features saved.")

print("\n" + "="*55)
print("Oxford Cats — Feature Extraction")
print("="*55)

cat_train_base, cat_train_black, cat_train_white, _ = \
    segment_and_extract(resnet18, cat_train, predictor, device, "Cat Train")

cat_val_base, cat_val_black, cat_val_white, _ = \
    segment_and_extract(resnet18, cat_val, predictor, device, "Cat Val")

# Save cats immediately
np.save(SAVE_DIR + 'oxford_cat_train_base.npy',  cat_train_base)
np.save(SAVE_DIR + 'oxford_cat_train_black.npy', cat_train_black)
np.save(SAVE_DIR + 'oxford_cat_train_white.npy', cat_train_white)
np.save(SAVE_DIR + 'oxford_cat_val_base.npy',    cat_val_base)
np.save(SAVE_DIR + 'oxford_cat_val_black.npy',   cat_val_black)
np.save(SAVE_DIR + 'oxford_cat_val_white.npy',   cat_val_white)
np.save(SAVE_DIR + 'oxford_cat_train_labels.npy', cat_train_labels)
np.save(SAVE_DIR + 'oxford_cat_val_labels.npy',   cat_val_labels)
print("Oxford Cats features saved.")

## Cell E — Normalize, cluster, and compare all pipelines:


In [ ]:
# ============================================
# CLUSTERING — OXFORD DOGS AND CATS
# ============================================

print("="*55)
print("Normalizing Oxford Dogs Features")
print("="*55)

dog_train_base_s,  dog_val_base_s,  _ = normalize(dog_train_base,  dog_val_base)
dog_train_black_s, dog_val_black_s, _ = normalize(dog_train_black, dog_val_black)
dog_train_white_s, dog_val_white_s, _ = normalize(dog_train_white, dog_val_white)

print("Normalizing Oxford Cats Features")
cat_train_base_s,  cat_val_base_s,  _ = normalize(cat_train_base,  cat_val_base)
cat_train_black_s, cat_val_black_s, _ = normalize(cat_train_black, cat_val_black)
cat_train_white_s, cat_val_white_s, _ = normalize(cat_train_white, cat_val_white)

print("\n" + "="*55)
print("Clustering — Oxford Dogs (25 breeds)")
print("="*55)

ox_dog_base  = run_kmeans(dog_train_base_s,  dog_val_base_s,  dog_train_labels, dog_val_labels, "Oxford Dogs Baseline",   n_breeds=25)
ox_dog_black = run_kmeans(dog_train_black_s, dog_val_black_s, dog_train_labels, dog_val_labels, "Oxford Dogs Black bg",    n_breeds=25)
ox_dog_white = run_kmeans(dog_train_white_s, dog_val_white_s, dog_train_labels, dog_val_labels, "Oxford Dogs White bg",    n_breeds=25)

print("\n" + "="*55)
print("Clustering — Oxford Cats (12 breeds)")
print("="*55)

ox_cat_base  = run_kmeans(cat_train_base_s,  cat_val_base_s,  cat_train_labels, cat_val_labels, "Oxford Cats Baseline",   n_breeds=12)
ox_cat_black = run_kmeans(cat_train_black_s, cat_val_black_s, cat_train_labels, cat_val_labels, "Oxford Cats Black bg",    n_breeds=12)
ox_cat_white = run_kmeans(cat_train_white_s, cat_val_white_s, cat_train_labels, cat_val_labels, "Oxford Cats White bg",    n_breeds=12)

# ============================================
# FINAL COMPARISON TABLE — ALL DATASETS
# ============================================

print("\n" + "="*80)
print("Final Comparison — All Datasets and Pipelines")
print("="*80)
print(f"{'Dataset':<20} {'Pipeline':<25} {'Breeds':>6} {'Best K':>6} {'ARI':>8} {'NMI':>8}")
print("-"*80)

all_results = [
    ("Imagewoof",     results_base,    10),
    ("Imagewoof",     results_black,   10),
    ("Imagewoof",     results_white,   10),
    ("Oxford Dogs",   ox_dog_base,     25),
    ("Oxford Dogs",   ox_dog_black,    25),
    ("Oxford Dogs",   ox_dog_white,    25),
    ("Oxford Cats",   ox_cat_base,     12),
    ("Oxford Cats",   ox_cat_black,    12),
    ("Oxford Cats",   ox_cat_white,    12),
]

for dataset, r, n_breeds in all_results:
    print(f"{dataset:<20} {r['name']:<25} {n_breeds:>6} {r['best_k']:>6} {r['ari']:>8.4f} {r['nmi']:>8.4f}")

print("="*80)

# Save final comparison
import pandas as pd
final_df = pd.DataFrame([
    {
        'Dataset'  : dataset,
        'Pipeline' : r['name'],
        'Breeds'   : n_breeds,
        'Best K'   : r['best_k'],
        'ARI'      : round(r['ari'], 4),
        'NMI'      : round(r['nmi'], 4),
        'Silhouette': round(r['silhouette'], 4),
    }
    for dataset, r, n_breeds in all_results
])
final_df.to_csv(SAVE_DIR + 'final_comparison_all_datasets.csv', index=False)
print("\nSaved: final_comparison_all_datasets.csv")

## Cell F — t-SNE visualizations for Oxford Dogs and Cats:

In [ ]:
# ============================================
# t-SNE VISUALIZATION — OXFORD DOGS
# ============================================

print("Running t-SNE for Oxford Dogs Baseline...")
plot_tsne(
    dog_train_base_s, dog_train_labels, ox_dog_base['train_clusters'],
    dog_readable, ox_dog_base['best_k'],
    "Oxford Dogs Baseline", SAVE_DIR + 'tsne_oxford_dogs_baseline.png'
)

plot_tsne(
    dog_train_black_s, dog_train_labels, ox_dog_black['train_clusters'],
    dog_readable, ox_dog_black['best_k'],
    "Oxford Dogs Black Background", SAVE_DIR + 'tsne_oxford_dogs_black.png'
)

plot_tsne(
    dog_train_white_s, dog_train_labels, ox_dog_white['train_clusters'],
    dog_readable, ox_dog_white['best_k'],
    "Oxford Dogs White Background", SAVE_DIR + 'tsne_oxford_dogs_white.png'
)

# ============================================
# t-SNE VISUALIZATION — OXFORD CATS
# ============================================

print("Running t-SNE for Oxford Cats Baseline...")
plot_tsne(
    cat_train_base_s, cat_train_labels, ox_cat_base['train_clusters'],
    cat_readable, ox_cat_base['best_k'],
    "Oxford Cats Baseline", SAVE_DIR + 'tsne_oxford_cats_baseline.png'
)

plot_tsne(
    cat_train_black_s, cat_train_labels, ox_cat_black['train_clusters'],
    cat_readable, ox_cat_black['best_k'],
    "Oxford Cats Black Background", SAVE_DIR + 'tsne_oxford_cats_black.png'
)

plot_tsne(
    cat_train_white_s, cat_train_labels, ox_cat_white['train_clusters'],
    cat_readable, ox_cat_white['best_k'],
    "Oxford Cats White Background", SAVE_DIR + 'tsne_oxford_cats_white.png'
)

## Cell G — Cluster sample images for Oxford Dogs and Cats:

In [ ]:
# ============================================
# CLUSTER SAMPLES — OXFORD DOGS AND CATS
# ============================================

plot_cluster_samples(
    dog_train, ox_dog_base['train_clusters'], dog_train_labels,
    dog_readable, ox_dog_base['best_k'],
    "Oxford Dogs Baseline", SAVE_DIR + 'samples_oxford_dogs_baseline.png'
)

plot_cluster_samples(
    cat_train, ox_cat_base['train_clusters'], cat_train_labels,
    cat_readable, ox_cat_base['best_k'],
    "Oxford Cats Baseline", SAVE_DIR + 'samples_oxford_cats_baseline.png'
)

## Cell H — Purity analysis for Oxford Dogs and Cats:


In [ ]:
# ============================================
# PURITY ANALYSIS — OXFORD DOGS AND CATS
# ============================================

print("Oxford Dogs Purity:")
cm_ox_dog_base  = purity_analysis(
    ox_dog_base['train_clusters'],  dog_train_labels,
    dog_readable, "Oxford Dogs Baseline", ox_dog_base['best_k']
)
cm_ox_dog_black = purity_analysis(
    ox_dog_black['train_clusters'], dog_train_labels,
    dog_readable, "Oxford Dogs Black bg", ox_dog_black['best_k']
)
cm_ox_dog_white = purity_analysis(
    ox_dog_white['train_clusters'], dog_train_labels,
    dog_readable, "Oxford Dogs White bg", ox_dog_white['best_k']
)

print("\nOxford Cats Purity:")
cm_ox_cat_base  = purity_analysis(
    ox_cat_base['train_clusters'],  cat_train_labels,
    cat_readable, "Oxford Cats Baseline", ox_cat_base['best_k']
)
cm_ox_cat_black = purity_analysis(
    ox_cat_black['train_clusters'], cat_train_labels,
    cat_readable, "Oxford Cats Black bg", ox_cat_black['best_k']
)
cm_ox_cat_white = purity_analysis(
    ox_cat_white['train_clusters'], cat_train_labels,
    cat_readable, "Oxford Cats White bg", ox_cat_white['best_k']
)

# ============================================
# CONFUSION MATRIX HEATMAPS
# ============================================

plot_confusion_heatmap(
    cm_ox_dog_base, dog_readable, ox_dog_base['best_k'],
    "Oxford Dogs Baseline", SAVE_DIR + 'cm_oxford_dogs_baseline.png'
)

plot_confusion_heatmap(
    cm_ox_cat_base, cat_readable, ox_cat_base['best_k'],
    "Oxford Cats Baseline", SAVE_DIR + 'cm_oxford_cats_baseline.png'
)

## Cell I — Cross-dataset summary comparison plot:


In [ ]:
# ============================================
# SUMMARY BAR CHART — ALL DATASETS
# ============================================

import matplotlib.pyplot as plt
import numpy as np

datasets   = ['Imagewoof\n(10 breeds)', 'Oxford Dogs\n(25 breeds)', 'Oxford Cats\n(12 breeds)']
ari_base   = [0.5800, 0.5296, 0.2608]
ari_black  = [0.2510, 0.2483, 0.1426]
ari_white  = [0.1434, 0.3620, 0.2612]

nmi_base   = [0.6897, 0.7885, 0.4815]
nmi_black  = [0.4113, 0.5385, 0.3158]
nmi_white  = [0.2538, 0.6139, 0.4510]

x = np.arange(len(datasets))
width = 0.25

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors = ['#2E5496', '#C55A11', '#538135']

for ax, metric_base, metric_black, metric_white, metric_name in [
    (axes[0], ari_base, ari_black, ari_white, 'ARI'),
    (axes[1], nmi_base, nmi_black, nmi_white, 'NMI'),
]:
    bars1 = ax.bar(x - width, metric_base,  width, label='Baseline',       color=colors[0], alpha=0.85)
    bars2 = ax.bar(x,         metric_black, width, label='SAM + Black bg',  color=colors[1], alpha=0.85)
    bars3 = ax.bar(x + width, metric_white, width, label='SAM + White bg',  color=colors[2], alpha=0.85)

    # Value labels on bars
    for bars in [bars1, bars2, bars3]:
        for bar in bars:
            height = bar.get_height()
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                height + 0.01, f'{height:.2f}',
                ha='center', va='bottom', fontsize=8
            )

    ax.set_xlabel('Dataset', fontsize=12)
    ax.set_ylabel(metric_name, fontsize=12)
    ax.set_title(f'{metric_name} Across All Datasets and Pipelines', fontsize=13, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(datasets, fontsize=10)
    ax.set_ylim(0, 1.0)
    ax.legend(fontsize=10)
    ax.axhline(y=0, color='black', linewidth=0.5)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle(
    'ResNet-18 + K-Means Clustering Performance\nBaseline vs SAM Background Removal',
    fontsize=14, fontweight='bold'
)
plt.tight_layout()
plt.savefig(SAVE_DIR + 'summary_comparison_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: summary_comparison_chart.png")

## Cell J: Controlled Lighting

## Experiment 2 : Controlled Lighting

In [ ]:
# ============================================
# EXPERIMENT 2 — CONTROLLED LIGHTING
# ============================================

import torchvision.transforms as transforms
from torch.utils.data import DataLoader, ConcatDataset, Subset
import torch
import numpy as np

print("="*60)
print("Experiment 2: Controlled Lighting")
print("="*60)

def make_dark_transform(brightness_factor):
    return transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ColorJitter(brightness=(brightness_factor, brightness_factor)),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])


def load_oxford_pets_darkened(brightness_factor):
    import torchvision.datasets as datasets
    dark_transform = make_dark_transform(brightness_factor)

    full = datasets.OxfordIIITPet(
        root='./data', split='trainval',
        target_types='category',
        transform=dark_transform, download=False
    )
    test = datasets.OxfordIIITPet(
        root='./data', split='test',
        target_types='category',
        transform=dark_transform, download=False
    )

    all_ds = ConcatDataset([full, test])
    all_lbls = np.array(
        [full[i][1] for i in range(len(full))] +
        [test[i][1]  for i in range(len(test))]
    )

    dog_idx = np.where(np.isin(all_lbls, dog_label_indices))[0]
    cat_idx = np.where(np.isin(all_lbls, cat_label_indices))[0]

    dog_tr, dog_v, dog_te = split_species_dataset(all_ds, dog_idx, dog_label_map, seed=42)
    cat_tr, cat_v, cat_te = split_species_dataset(all_ds, cat_idx, cat_label_map, seed=42)

    return dog_tr, dog_v, cat_tr, cat_v


def extract_baseline_only(model, dataset, device, description="Extracting"):
    model.eval()
    features_list = []
    labels_list   = []

    from tqdm import tqdm
    with torch.no_grad():
        for idx in tqdm(range(len(dataset)), desc=description):
            image, label = dataset[idx]
            feat = model(image.unsqueeze(0).to(device))
            feat = feat.view(1, -1).cpu().numpy()
            features_list.append(feat)
            labels_list.append(label)

    return np.concatenate(features_list), np.array(labels_list)


# ---- Reload existing features from Drive if needed ----
dog_train_base   = np.load(SAVE_DIR + 'oxford_dog_train_base.npy')
dog_val_base     = np.load(SAVE_DIR + 'oxford_dog_val_base.npy')
cat_train_base   = np.load(SAVE_DIR + 'oxford_cat_train_base.npy')
cat_val_base     = np.load(SAVE_DIR + 'oxford_cat_val_base.npy')
dog_train_labels = np.load(SAVE_DIR + 'oxford_dog_train_labels.npy')
dog_val_labels   = np.load(SAVE_DIR + 'oxford_dog_val_labels.npy')
cat_train_labels = np.load(SAVE_DIR + 'oxford_cat_train_labels.npy')
cat_val_labels   = np.load(SAVE_DIR + 'oxford_cat_val_labels.npy')
print("Existing features loaded from Drive.")


# ---- Run for each brightness level ----
brightness_levels = {
    'normal'    : 1.0,
    'dark'      : 0.4,
    'very_dark' : 0.2,
}

lighting_results = {}

for level_name, factor in brightness_levels.items():
    print(f"\n{'='*55}")
    print(f"Brightness level: {level_name} (factor={factor})")
    print(f"{'='*55}")

    if level_name == 'normal':
        dog_tr_feat, dog_tr_lbl = dog_train_base,   dog_train_labels
        dog_v_feat,  dog_v_lbl  = dog_val_base,     dog_val_labels
        cat_tr_feat, cat_tr_lbl = cat_train_base,   cat_train_labels
        cat_v_feat,  cat_v_lbl  = cat_val_base,     cat_val_labels

    else:
        dog_tr_dark, dog_v_dark, cat_tr_dark, cat_v_dark = \
            load_oxford_pets_darkened(factor)

        dog_tr_feat, _ = extract_baseline_only(
            resnet18, dog_tr_dark, device, f"Dog Train ({level_name})"
        )
        dog_v_feat, _ = extract_baseline_only(
            resnet18, dog_v_dark, device, f"Dog Val ({level_name})"
        )
        cat_tr_feat, _ = extract_baseline_only(
            resnet18, cat_tr_dark, device, f"Cat Train ({level_name})"
        )
        cat_v_feat, _ = extract_baseline_only(
            resnet18, cat_v_dark, device, f"Cat Val ({level_name})"
        )

        dog_tr_lbl = dog_train_labels
        dog_v_lbl  = dog_val_labels
        cat_tr_lbl = cat_train_labels
        cat_v_lbl  = cat_val_labels

        np.save(SAVE_DIR + f'oxford_dog_train_{level_name}.npy', dog_tr_feat)
        np.save(SAVE_DIR + f'oxford_dog_val_{level_name}.npy',   dog_v_feat)
        np.save(SAVE_DIR + f'oxford_cat_train_{level_name}.npy', cat_tr_feat)
        np.save(SAVE_DIR + f'oxford_cat_val_{level_name}.npy',   cat_v_feat)
        print(f"Saved {level_name} features.")

    # Normalize
    dog_tr_s, dog_v_s, _ = normalize(dog_tr_feat, dog_v_feat)
    cat_tr_s, cat_v_s, _ = normalize(cat_tr_feat, cat_v_feat)

    # Cluster
    dog_res = run_kmeans(
        dog_tr_s, dog_v_s, dog_tr_lbl, dog_v_lbl,
        f"Oxford Dogs {level_name}", n_breeds=25
    )
    cat_res = run_kmeans(
        cat_tr_s, cat_v_s, cat_tr_lbl, cat_v_lbl,
        f"Oxford Cats {level_name}", n_breeds=12
    )

    lighting_results[level_name] = {
        'dog': dog_res,
        'cat': cat_res
    }


# ---- Results Table ----
print("\n" + "="*75)
print("Experiment 2 Results — Effect of Lighting on Clustering")
print("="*75)
print(f"{'Brightness':<12} {'Species':<10} {'Best K':>6} {'ARI':>8} {'NMI':>8}  Degradation")
print("-"*75)

baseline_dog_ari = lighting_results['normal']['dog']['ari']
baseline_cat_ari = lighting_results['normal']['cat']['ari']

for level in ['normal', 'dark', 'very_dark']:
    for species, baseline_ari in [('dog', baseline_dog_ari), ('cat', baseline_cat_ari)]:
        r = lighting_results[level][species]
        drop = r['ari'] - baseline_ari
        drop_str = f"{drop:+.4f}" if level != 'normal' else "baseline"
        print(f"{level:<12} {species:<10} {r['best_k']:>6} {r['ari']:>8.4f} {r['nmi']:>8.4f}  {drop_str}")

print("="*75)

import pandas as pd
lighting_df = pd.DataFrame([
    {
        'Brightness'  : level,
        'Species'     : species,
        'Best K'      : lighting_results[level][species]['best_k'],
        'ARI'         : round(lighting_results[level][species]['ari'], 4),
        'NMI'         : round(lighting_results[level][species]['nmi'], 4),
    }
    for level in ['normal', 'dark', 'very_dark']
    for species in ['dog', 'cat']
])
lighting_df.to_csv(SAVE_DIR + 'experiment2_lighting.csv', index=False)
print("Saved: experiment2_lighting.csv")

## Experiment 3: ViT FEATURE EXTRACTORS

## Cell L — Load ViT-B/16 and ViT-L/16:

In [ ]:
# ============================================
# EXPERIMENT 3 — ViT FEATURE EXTRACTORS
# ============================================

import torchvision.models as models
import torch

print("="*60)
print("Experiment 3: ViT vs ResNet Feature Extractors")
print("="*60)

class ViTFeatureExtractor(torch.nn.Module):
    def __init__(self, vit_model):
        super().__init__()
        self.model = vit_model

    def forward(self, x):
        x = self.model._process_input(x)
        n = x.shape[0]
        batch_class_token = self.model.class_token.expand(n, -1, -1)
        x = torch.cat([batch_class_token, x], dim=1)
        x = self.model.encoder(x)
        return x[:, 0]  # return class token only

# ---- ViT-B/16 ----
print("\nLoading ViT-B/16...")
vit_b = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1)
vit_b_extractor = ViTFeatureExtractor(vit_b)
vit_b_extractor.eval()
vit_b_extractor = vit_b_extractor.to(device)

dummy = torch.randn(1, 3, 224, 224).to(device)
with torch.no_grad():
    out = vit_b_extractor(dummy)
print(f"ViT-B/16 output shape : {out.shape}  (expected [1, 768])")

# ---- ViT-L/16 ----
print("\nLoading ViT-L/16...")
vit_l = models.vit_l_16(weights=models.ViT_L_16_Weights.IMAGENET1K_V1)
vit_l_extractor = ViTFeatureExtractor(vit_l)
vit_l_extractor.eval()
vit_l_extractor = vit_l_extractor.to(device)

with torch.no_grad():
    out = vit_l_extractor(dummy)
print(f"ViT-L/16 output shape : {out.shape}  (expected [1, 1024])")

print("\nBoth ViT models loaded and ready.")

## Cell M — Extract ViT features:

In [ ]:
# ============================================
# EXPERIMENT 3 — EXTRACT ViT FEATURES
# ============================================

def extract_vit_features(model, dataset, device, description="Extracting"):
    model.eval()
    features_list = []
    labels_list   = []

    from tqdm import tqdm
    with torch.no_grad():
        for idx in tqdm(range(len(dataset)), desc=description):
            image, label = dataset[idx]
            feat = model(image.unsqueeze(0).to(device))
            feat = feat.view(1, -1).cpu().numpy()
            features_list.append(feat)
            labels_list.append(label)

    return np.concatenate(features_list), np.array(labels_list)


# ---- ViT-B/16 ----
print("="*55)
print("ViT-B/16 Feature Extraction")
print("="*55)

vitb_dog_train_feat, _ = extract_vit_features(
    vit_b_extractor, dog_train, device, "ViT-B Dog Train"
)
vitb_dog_val_feat, _ = extract_vit_features(
    vit_b_extractor, dog_val, device, "ViT-B Dog Val"
)
vitb_cat_train_feat, _ = extract_vit_features(
    vit_b_extractor, cat_train, device, "ViT-B Cat Train"
)
vitb_cat_val_feat, _ = extract_vit_features(
    vit_b_extractor, cat_val, device, "ViT-B Cat Val"
)

np.save(SAVE_DIR + 'vitb_dog_train.npy', vitb_dog_train_feat)
np.save(SAVE_DIR + 'vitb_dog_val.npy',   vitb_dog_val_feat)
np.save(SAVE_DIR + 'vitb_cat_train.npy', vitb_cat_train_feat)
np.save(SAVE_DIR + 'vitb_cat_val.npy',   vitb_cat_val_feat)
print(f"\nViT-B/16 features saved.")
print(f"  Dog train : {vitb_dog_train_feat.shape}")
print(f"  Cat train : {vitb_cat_train_feat.shape}")

# ---- ViT-L/16 ----
print("\n" + "="*55)
print("ViT-L/16 Feature Extraction")
print("="*55)

vitl_dog_train_feat, _ = extract_vit_features(
    vit_l_extractor, dog_train, device, "ViT-L Dog Train"
)
vitl_dog_val_feat, _ = extract_vit_features(
    vit_l_extractor, dog_val, device, "ViT-L Dog Val"
)
vitl_cat_train_feat, _ = extract_vit_features(
    vit_l_extractor, cat_train, device, "ViT-L Cat Train"
)
vitl_cat_val_feat, _ = extract_vit_features(
    vit_l_extractor, cat_val, device, "ViT-L Cat Val"
)

np.save(SAVE_DIR + 'vitl_dog_train.npy', vitl_dog_train_feat)
np.save(SAVE_DIR + 'vitl_dog_val.npy',   vitl_dog_val_feat)
np.save(SAVE_DIR + 'vitl_cat_train.npy', vitl_cat_train_feat)
np.save(SAVE_DIR + 'vitl_cat_val.npy',   vitl_cat_val_feat)
print(f"\nViT-L/16 features saved.")
print(f"  Dog train : {vitl_dog_train_feat.shape}")
print(f"  Cat train : {vitl_cat_train_feat.shape}")

print(f"\nFeature dimensions summary:")
print(f"  ResNet-18 : 512")
print(f"  ViT-B/16  : {vitb_dog_train_feat.shape[1]}")
print(f"  ViT-L/16  : {vitl_dog_train_feat.shape[1]}")



## Cell N — Cluster and compare:



In [ ]:
# Recovery — recompute ResNet baseline clustering from saved features
dog_tr_s, dog_v_s, _ = normalize(dog_train_base, dog_val_base)
cat_tr_s, cat_v_s, _ = normalize(cat_train_base, cat_val_base)

ox_dog_base = run_kmeans(
    dog_tr_s, dog_v_s, dog_train_labels, dog_val_labels,
    "Oxford Dogs Baseline", n_breeds=25
)
ox_cat_base = run_kmeans(
    cat_tr_s, cat_v_s, cat_train_labels, cat_val_labels,
    "Oxford Cats Baseline", n_breeds=12
)

In [ ]:
# ============================================
# EXPERIMENT 3 — CLUSTER AND COMPARE
# ============================================

# Normalize
vitb_dog_tr_s, vitb_dog_v_s, _ = normalize(vitb_dog_train_feat, vitb_dog_val_feat)
vitb_cat_tr_s, vitb_cat_v_s, _ = normalize(vitb_cat_train_feat, vitb_cat_val_feat)
vitl_dog_tr_s, vitl_dog_v_s, _ = normalize(vitl_dog_train_feat, vitl_dog_val_feat)
vitl_cat_tr_s, vitl_cat_v_s, _ = normalize(vitl_cat_train_feat, vitl_cat_val_feat)

# Cluster ViT-B
print("="*55)
print("Clustering ViT-B/16 Features")
print("="*55)

vitb_dog_res = run_kmeans(
    vitb_dog_tr_s, vitb_dog_v_s,
    dog_train_labels, dog_val_labels,
    "ViT-B Oxford Dogs", n_breeds=25
)
vitb_cat_res = run_kmeans(
    vitb_cat_tr_s, vitb_cat_v_s,
    cat_train_labels, cat_val_labels,
    "ViT-B Oxford Cats", n_breeds=12
)

# Cluster ViT-L
print("\n" + "="*55)
print("Clustering ViT-L/16 Features")
print("="*55)

vitl_dog_res = run_kmeans(
    vitl_dog_tr_s, vitl_dog_v_s,
    dog_train_labels, dog_val_labels,
    "ViT-L Oxford Dogs", n_breeds=25
)
vitl_cat_res = run_kmeans(
    vitl_cat_tr_s, vitl_cat_v_s,
    cat_train_labels, cat_val_labels,
    "ViT-L Oxford Cats", n_breeds=12
)

# ---- Comparison Table ----
print("\n" + "="*80)
print("Experiment 3 Results — Model Architecture Comparison")
print("="*80)
print(f"{'Model':<14} {'Dims':>5} {'Species':<10} {'Best K':>6} {'ARI':>8} {'NMI':>8}")
print("-"*80)

model_results = [
    ("ResNet-18",  512,  "Dogs", ox_dog_base),
    ("ResNet-18",  512,  "Cats", ox_cat_base),
    ("ViT-B/16",   768,  "Dogs", vitb_dog_res),
    ("ViT-B/16",   768,  "Cats", vitb_cat_res),
    ("ViT-L/16",   1024, "Dogs", vitl_dog_res),
    ("ViT-L/16",   1024, "Cats", vitl_cat_res),
]

for model_name, dims, species, r in model_results:
    print(f"{model_name:<14} {dims:>5} {species:<10} {r['best_k']:>6} {r['ari']:>8.4f} {r['nmi']:>8.4f}")

print("="*80)

# Save
import pandas as pd
vit_df = pd.DataFrame([
    {
        'Model'  : model_name,
        'Dims'   : dims,
        'Species': species,
        'Best K' : r['best_k'],
        'ARI'    : round(r['ari'], 4),
        'NMI'    : round(r['nmi'], 4),
    }
    for model_name, dims, species, r in model_results
])
vit_df.to_csv(SAVE_DIR + 'experiment3_vit_comparison.csv', index=False)
print("Saved: experiment3_vit_comparison.csv")

## Cell O — Visualization:

In [ ]:
# ============================================
# EXPERIMENT 3 — VISUALIZATION
# ============================================

import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

models_labels = ['ResNet-18\n(512d)', 'ViT-B/16\n(768d)', 'ViT-L/16\n(1024d)']
colors_model  = ['#2E5496', '#538135', '#7B2C2C']

dog_ari = [ox_dog_base['ari'], vitb_dog_res['ari'], vitl_dog_res['ari']]
dog_nmi = [ox_dog_base['nmi'], vitb_dog_res['nmi'], vitl_dog_res['nmi']]
cat_ari = [ox_cat_base['ari'], vitb_cat_res['ari'], vitl_cat_res['ari']]
cat_nmi = [ox_cat_base['nmi'], vitb_cat_res['nmi'], vitl_cat_res['nmi']]

for ax, ari_vals, nmi_vals, title in [
    (axes[0], dog_ari, dog_nmi, 'Oxford Dogs (25 breeds)'),
    (axes[1], cat_ari, cat_nmi, 'Oxford Cats (12 breeds)'),
]:
    x = np.arange(len(models_labels))
    width = 0.35

    bars1 = ax.bar(x - width/2, ari_vals, width,
                   label='ARI', color=colors_model, alpha=0.85)
    bars2 = ax.bar(x + width/2, nmi_vals, width,
                   label='NMI', color=colors_model, alpha=0.5, hatch='//')

    for bar in bars1:
        ax.text(
            bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.01,
            f'{bar.get_height():.3f}',
            ha='center', va='bottom', fontsize=9
        )
    for bar in bars2:
        ax.text(
            bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.01,
            f'{bar.get_height():.3f}',
            ha='center', va='bottom', fontsize=9
        )

    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(models_labels, fontsize=10)
    ax.set_ylim(0, 1.0)
    ax.set_ylabel('Score', fontsize=11)
    ax.legend(fontsize=10)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle(
    'Experiment 3: ResNet-18 vs ViT-B/16 vs ViT-L/16\nFeature Extractor Comparison on Oxford Pets',
    fontsize=14, fontweight='bold'
)
plt.tight_layout()
plt.savefig(SAVE_DIR + 'experiment3_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: experiment3_model_comparison.png")